# Hemocytometer Crystal Analysis Pipeline

Automated pipeline for detecting and sizing protein crystals in hemocytometer images.
Developed at SLAC National Accelerator Laboratory (LCLS / MFX instrument) under Dr. Elyse Schriber.

## Pipeline Overview

1. **Grid alignment** — Canny edge detection + HoughLinesP to detect and auto-rotate the hemocytometer grid
2. **Pixel-to-mm calibration** — DBSCAN clustering of grid lines to measure physical scale
3. **Cell cropping** — Mask grid lines, extract individual well rectangles, calculate cell volumes
4. **Crystal segmentation** — Custom "ant eating" flood-fill algorithm with multi-threshold stop zones
5. **Backbone segmentation** *(work in progress)* — Skeletonization to split overlapping crystals

## Requirements
```
pip install opencv-python numpy matplotlib scikit-learn scikit-image
```

## Step 1 — Grid Alignment

Loads the hemocytometer image, detects Canny edges with adaptive thresholds, and uses HoughLinesP to find the grid. Lines are clustered with DBSCAN to average out duplicates, then the image is rotated to align the grid horizontally/vertically.

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN

# ── Config ──
IMG_FOLDER = "data/images"   # folder containing hemocytometer .jpg images
SHOW_DEBUG = True            # set False for minimal output

# ── Load image ──
img_files = sorted([f for f in os.listdir(IMG_FOLDER) if f.lower().endswith((".jpg", ".png"))])
if not img_files:
    raise FileNotFoundError(f"No images found in {IMG_FOLDER}")

print("Available images:")
for i, name in enumerate(img_files, 1):
    print(f"  {i}. {name}")

selection = input(f"\nSelect image (1-{len(img_files)}): ").strip()
img_name = img_files[int(selection) - 1]
img = cv2.imread(os.path.join(IMG_FOLDER, img_name))
print(f"Loaded: {img_name}  ({img.shape[1]}x{img.shape[0]} px)")

if SHOW_DEBUG:
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(f"Input: {img_name}"); plt.axis("off"); plt.show()

# ── Adaptive Canny ──
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
median_val = np.median(gray)
edges = cv2.Canny(gray,
                  int(max(0,   0.66 * median_val)),
                  int(min(255, 1.33 * median_val)))

# ── Detect rotation angle from Hough lines ──
raw_lines = cv2.HoughLinesP(edges, 1, np.pi/180, threshold=100,
                             minLineLength=100, maxLineGap=10)
if raw_lines is None:
    raise RuntimeError("No lines detected — try a clearer image.")

angles = []
for line in raw_lines:
    x1, y1, x2, y2 = line[0]
    a = np.arctan2(y2-y1, x2-x1) * 180 / np.pi
    if a > 90:  a -= 180
    if a < -90: a += 180
    if abs(a) < 20 or abs(abs(a) - 90) < 20:
        angles.append(a)

if not angles:
    raise RuntimeError("No dominant grid lines found for rotation.")

angles = np.array(angles)
median_angle = np.median(angles)
rotation_angle = np.mean(angles[np.abs(angles - median_angle) < 5])
print(f"Rotation angle: {rotation_angle:.2f}°")

# ── Rotate image ──
h, w = img.shape[:2]
M = cv2.getRotationMatrix2D((w//2, h//2), rotation_angle, 1.0)
rotated_img = cv2.warpAffine(img, M, (w, h), flags=cv2.INTER_LINEAR)
rotated_gray = cv2.cvtColor(rotated_img, cv2.COLOR_BGR2GRAY)

if SHOW_DEBUG:
    plt.imshow(cv2.cvtColor(rotated_img, cv2.COLOR_BGR2RGB))
    plt.title(f"Rotated ({rotation_angle:.2f}°)"); plt.axis("off"); plt.show()

# ── Cluster Hough lines to get clean grid ──
def average_lines_by_position(lines, orientation, eps=50, min_cluster_size=2, image_shape=None):
    """DBSCAN clustering to merge duplicate Hough lines into single averaged lines."""
    if lines is None:
        return []
    coords = []
    for line in lines:
        x1, y1, x2, y2 = line[0]
        if orientation == "horizontal" and abs(y2-y1) < abs(x2-x1):
            coords.append((y1+y2)//2)
        elif orientation == "vertical" and abs(x2-x1) < abs(y2-y1):
            coords.append((x1+x2)//2)
    if not coords:
        return []
    coords = np.array(coords).reshape(-1, 1)
    labels = DBSCAN(eps=eps, min_samples=1).fit(coords).labels_
    averaged = []
    for label in np.unique(labels):
        members = coords[labels == label]
        if len(members) < min_cluster_size:
            continue
        avg = int(np.mean(members))
        if orientation == "horizontal":
            averaged.append((0, avg, image_shape[1], avg))
        else:
            averaged.append((avg, 0, avg, image_shape[0]))
    return averaged

median_rot = np.median(rotated_gray)
rotated_edges = cv2.Canny(rotated_gray,
                           int(max(0,   0.66 * median_rot)),
                           int(min(255, 1.33 * median_rot)))
raw_lines_rot = cv2.HoughLinesP(rotated_edges, 1, np.pi/180, threshold=100,
                                  minLineLength=100, maxLineGap=10)

avg_h_lines = average_lines_by_position(raw_lines_rot, "horizontal", image_shape=rotated_img.shape)
avg_v_lines = average_lines_by_position(raw_lines_rot, "vertical",   image_shape=rotated_img.shape)
print(f"Grid lines detected: {len(avg_h_lines)} horizontal, {len(avg_v_lines)} vertical")

if SHOW_DEBUG:
    debug = rotated_img.copy()
    for x1, y1, x2, y2 in avg_h_lines + avg_v_lines:
        cv2.line(debug, (x1,y1), (x2,y2), (0,255,0), 1)
    plt.imshow(cv2.cvtColor(debug, cv2.COLOR_BGR2RGB))
    plt.title("Averaged Grid Lines"); plt.axis("off"); plt.show()

## Step 2 — Pixel-to-mm Calibration

Measures average spacing between detected grid lines. Each hemocytometer grid square = 0.05 mm, so spacing gives the mm/pixel scale factor. Chamber depth is fixed at 0.1 mm.

In [ ]:
def measure_spacing(lines, orientation="horizontal"):
    """Returns average pixel spacing and list of individual spacings between grid lines."""
    if not lines:
        return None, None
    coords = sorted(set([y1 for _,y1,_,_ in lines] if orientation == "horizontal"
                        else [x1 for x1,_,_,_ in lines]))
    spacings = [coords[i+1] - coords[i] for i in range(len(coords)-1)]
    return np.mean(spacings), spacings

avg_h_spacing, h_spacings = measure_spacing(avg_h_lines, "horizontal")
avg_v_spacing, v_spacings = measure_spacing(avg_v_lines, "vertical")

if avg_h_spacing is None or avg_v_spacing is None:
    raise RuntimeError("Could not measure grid spacing — check line detection.")

# Each hemocytometer grid square = 0.05 mm
mm_per_pixel_h   = 0.05 / avg_h_spacing
mm_per_pixel_v   = 0.05 / avg_v_spacing
mm_per_pixel_avg = (mm_per_pixel_h + mm_per_pixel_v) / 2

print(f"Horizontal spacing: {avg_h_spacing:.2f} px  →  {mm_per_pixel_h:.6f} mm/px")
print(f"Vertical spacing:   {avg_v_spacing:.2f} px  →  {mm_per_pixel_v:.6f} mm/px")
print(f"Scale (avg):        {mm_per_pixel_avg:.6f} mm/px")

if SHOW_DEBUG:
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    for ax, spacings, label, color in zip(
        axes,
        [h_spacings, v_spacings],
        ["Horizontal spacings (px)", "Vertical spacings (px)"],
        ["steelblue", "seagreen"]
    ):
        avg = np.mean(spacings)
        ax.hist(spacings, bins=10, color=color, alpha=0.7)
        ax.axvline(avg, color="red", linestyle="--", label=f"Avg: {avg:.1f} px")
        ax.set_title(label); ax.legend()
    plt.tight_layout(); plt.show()

## Step 3 — Cell Extraction & Volume Calculation

Masks the grid lines, finds individual well rectangles via contour detection, filters out edge/dark cells, then randomly samples 10 cells for analysis. Calculates each cell's physical volume using pixel-to-mm scale and hemocytometer depth (0.1 mm).

In [ ]:
import random

# ── Build grid line mask ──
line_thickness = int(max(3, round((avg_h_spacing + avg_v_spacing) / 5)))
grid_mask = np.zeros(rotated_gray.shape, dtype=np.uint8)
for x1, y1, x2, y2 in avg_h_lines + avg_v_lines:
    cv2.line(grid_mask, (x1,y1), (x2,y2), 255, thickness=line_thickness)

# ── Find cell rectangles from inverted mask ──
inv_mask = cv2.bitwise_not(grid_mask)
contours, _ = cv2.findContours(inv_mask, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
h_img, w_img = rotated_gray.shape

def has_dark_border(x, y, w, h, gray_img, border_width=5, dark_thresh=30, frac=0.3):
    """Returns True if any edge strip of the ROI is mostly black (grid border artefact)."""
    roi = gray_img[y:y+h, x:x+w]
    for side in [roi[:border_width,:], roi[-border_width:,:],
                 roi[:,:border_width], roi[:,-border_width:]]:
        if (np.sum(side < dark_thresh) / side.size) > frac:
            return True
    return False

filtered_rects = []
for cnt in contours:
    x, y, w, h = cv2.boundingRect(cnt)
    if x <= 0 or y <= 0 or (x+w) >= w_img or (y+h) >= h_img:
        continue
    if has_dark_border(x, y, w, h, rotated_gray):
        continue
    filtered_rects.append((x, y, w, h))

print(f"Valid cells found: {len(filtered_rects)}")

# ── Sample 10 random cells ──
picked_rects = random.sample(filtered_rects, min(10, len(filtered_rects)))

# ── Volume calculation ──
# Hemocytometer chamber depth = 0.1 mm (standard)
print(f"\nSampled {len(picked_rects)} cells:")
for i, (x, y, w, h) in enumerate(picked_rects, 1):
    width_mm  = w * mm_per_pixel_avg
    height_mm = h * mm_per_pixel_avg
    volume    = width_mm * height_mm * 0.1
    print(f"  Cell {i}: {width_mm:.4f} x {height_mm:.4f} mm  →  volume = {volume:.6f} mm³")

if SHOW_DEBUG:
    debug = rotated_img.copy()
    for i, (x, y, w, h) in enumerate(picked_rects, 1):
        cv2.rectangle(debug, (x,y), (x+w,y+h), (0,255,255), 2)
        cv2.putText(debug, str(i), (x+3, y+15),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,0,255), 1)
    plt.imshow(cv2.cvtColor(debug, cv2.COLOR_BGR2RGB))
    plt.title("Sampled Cells"); plt.axis("off"); plt.show()

## Step 4 — Crystal Segmentation ("Ant Eating")

Custom flood-fill segmentation algorithm. Rather than simple thresholding, the algorithm:

1. Builds a **voted edge mask** from Canny + Scharr + morphological gradient (pixel must be detected by ≥2 methods)
2. Defines **stop zones** — dark pixels and strong edges where the flood-fill cannot cross
3. Floods inward from image borders and bright interior holes, erasing non-crystal background pixels
4. Repeats for multiple passes until convergence
5. Applies watershed on the remaining mask for instance separation

The result is a per-cell binary mask and labeled crystal count.

In [ ]:
from collections import deque

def segment_crystals(crop_bgr,
                     dark_threshold=50,
                     strong_edge_threshold=80,
                     soft_stop_threshold=120,
                     bright_start_threshold=150,
                     max_passes=5,
                     show_debug=True):
    """
    Ant-eating flood-fill crystal segmentation.

    Parameters
    ----------
    dark_threshold          : pixels darker than this are treated as crystal edges (stop zones)
    strong_edge_threshold   : Scharr gradient magnitude threshold for edge stop zones
    soft_stop_threshold     : secondary soft boundary to slow flood-fill
    bright_start_threshold  : bright interior pixels used as additional flood-fill seeds
    max_passes              : maximum flood-fill iterations (stops early if converged)
    """
    gray   = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2GRAY)
    h, w   = gray.shape
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3,3))

    # Build stop zones: dark regions + strong edges
    dark_mask = (gray < dark_threshold).astype(np.uint8) * 255
    scharr_x  = cv2.Scharr(gray, cv2.CV_64F, 1, 0)
    scharr_y  = cv2.Scharr(gray, cv2.CV_64F, 0, 1)
    scharr_mag = cv2.normalize(cv2.magnitude(scharr_x, scharr_y),
                                None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    _, strong_edges = cv2.threshold(scharr_mag, strong_edge_threshold, 255, cv2.THRESH_BINARY)
    stop_zone      = cv2.bitwise_or(dark_mask, strong_edges)
    soft_stop_zone = (gray < soft_stop_threshold).astype(np.uint8) * 255

    # Voted edge mask: pixel must be detected by ≥2 of 3 methods
    edges_canny = cv2.Canny(gray, 30, 100)
    morph_grad  = cv2.morphologyEx(gray, cv2.MORPH_GRADIENT, kernel)
    _, morph_edges = cv2.threshold(morph_grad, 15, 255, cv2.THRESH_BINARY)
    votes = ((edges_canny > 0).astype(np.uint8) +
             (scharr_mag  > 0).astype(np.uint8) +
             (morph_edges > 0).astype(np.uint8))
    mask = (votes >= 2).astype(np.uint8) * 255
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=2)

    # Multi-pass flood-fill from borders + bright interior holes
    for _ in range(max_passes):
        before  = mask.copy()
        visited = np.zeros_like(gray, dtype=np.uint8)
        q       = deque()

        for x in range(w): q.extend([(0,x), (h-1,x)])
        for y in range(h): q.extend([(y,0), (y,w-1)])

        inv_mask    = cv2.bitwise_not(mask)
        bright_holes = cv2.bitwise_and(
            (gray > bright_start_threshold).astype(np.uint8) * 255, inv_mask)
        contours_bh, _ = cv2.findContours(bright_holes, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        for cnt in contours_bh:
            for pt in cnt:
                q.append((pt[0][1], pt[0][0]))

        while q:
            cy, cx = q.popleft()
            if not (0 <= cx < w and 0 <= cy < h): continue
            if visited[cy, cx]: continue
            visited[cy, cx] = 1
            if stop_zone[cy,cx] or soft_stop_zone[cy,cx]: continue
            mask[cy, cx] = 0
            for dy in [-1,0,1]:
                for dx in [-1,0,1]:
                    if dy == dx == 0: continue
                    ny, nx = cy+dy, cx+dx
                    if 0 <= nx < w and 0 <= ny < h and not visited[ny,nx]:
                        q.append((ny, nx))

        # Smooth large blobs only
        contours_m, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        areas = [cv2.contourArea(c) for c in contours_m]
        area_thresh = np.median(areas) if areas else 0
        for cnt in contours_m:
            if cv2.contourArea(cnt) >= area_thresh:
                tmp = np.zeros_like(mask)
                cv2.drawContours(tmp, [cnt], -1, 255, cv2.FILLED)
                tmp = cv2.morphologyEx(tmp, cv2.MORPH_OPEN,  kernel, iterations=1)
                tmp = cv2.morphologyEx(tmp, cv2.MORPH_CLOSE, kernel, iterations=2)
                mask[tmp > 0] = 255

        if np.array_equal(mask, before):
            break

    # Watershed for instance separation
    sure_bg = cv2.dilate(mask, None, iterations=2)
    dist    = cv2.distanceTransform(mask, cv2.DIST_L2, 5)
    _, sure_fg = cv2.threshold(dist, 0.4*dist.max(), 255, 0)
    sure_fg    = sure_fg.astype(np.uint8)
    unknown    = cv2.subtract(sure_bg, sure_fg)
    num_labels, markers = cv2.connectedComponents(sure_fg)
    markers = markers + 1
    markers[unknown == 255] = 0
    img_ws  = crop_bgr.copy()
    markers = cv2.watershed(img_ws, markers)

    overlay = crop_bgr.copy()
    overlay[markers == -1] = [0, 255, 0]
    crystal_count = num_labels - 1  # exclude background label

    if show_debug:
        fig, axes = plt.subplots(1, 4, figsize=(16, 4))
        axes[0].imshow(cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)); axes[0].set_title("Original")
        axes[1].imshow(mask, cmap="gray");                          axes[1].set_title("Final Mask")
        axes[2].imshow(dist, cmap="jet");                           axes[2].set_title("Distance Transform")
        axes[3].imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB));   axes[3].set_title(f"Overlay (Count: {crystal_count})")
        for ax in axes: ax.axis("off")
        plt.tight_layout(); plt.show()

    return mask, overlay, crystal_count


# ── Run on sampled cells ──
total_crystals = 0
for i, (x, y, w, h) in enumerate(picked_rects, 1):
    crop = rotated_img[y:y+h, x:x+w]
    print(f"Cell {i}:")
    mask, overlay, count = segment_crystals(crop, show_debug=SHOW_DEBUG)
    total_crystals += count

print(f"\nTotal crystals detected across {len(picked_rects)} cells: {total_crystals}")

## Step 5 — Backbone Segmentation *(Work in Progress)*

For touching or overlapping crystals, watershed alone is insufficient. This step uses **skeletonization** (`skimage.morphology.skeletonize`) to trace the skeleton/spine of each segmented blob, then identifies dark points along the skeleton as likely crystal boundaries. Cut lines are drawn through these points to split overlapping crystals.

This approach was under development and not fully validated — results are approximate.

In [ ]:
from skimage.morphology import skeletonize

def segment_with_backbone(crop_bgr,
                           dark_threshold=50,
                           backbone_dark_threshold=60,
                           show_debug=True):
    """
    Extends ant-eating segmentation with skeleton-based splitting of overlapping crystals.
    Skeletonizes each detected blob, finds dark points along the skeleton,
    and draws cut lines to separate touching crystals.
    """
    # Re-use ant-eating to get initial mask
    mask, _, _ = segment_crystals(crop_bgr, dark_threshold=dark_threshold,
                                   show_debug=False)
    gray         = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2GRAY)
    final_overlay = crop_bgr.copy()

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    for cnt in contours:
        if cv2.contourArea(cnt) < 10:
            continue
        x, y, wb, hb = cv2.boundingRect(cnt)
        blob_mask = np.zeros_like(mask)
        cv2.drawContours(blob_mask, [cnt], -1, 255, -1)
        blob_gray  = gray[y:y+hb, x:x+wb]
        blob_mask_crop = blob_mask[y:y+hb, x:x+wb]

        # Skeletonize
        skel = skeletonize((blob_mask_crop > 0).astype(np.uint8))

        # Dark points along skeleton = likely inter-crystal boundaries
        dark_pts = np.logical_and(skel, blob_gray < backbone_dark_threshold)

        # Draw cut lines through dark skeleton points
        blob_overlay = crop_bgr[y:y+hb, x:x+wb].copy()
        for py, px in np.argwhere(dark_pts):
            cv2.line(blob_overlay, (px,  0), (px, hb-1), (0,0,255), 1)
            cv2.line(blob_overlay, (0,  py), (wb-1, py), (0,0,255), 1)
        final_overlay[y:y+hb, x:x+wb] = blob_overlay

        if show_debug:
            fig, axes = plt.subplots(1, 4, figsize=(16, 4))
            axes[0].imshow(cv2.cvtColor(crop_bgr[y:y+hb,x:x+wb], cv2.COLOR_BGR2RGB)); axes[0].set_title("Blob")
            axes[1].imshow(blob_mask_crop, cmap="gray");  axes[1].set_title("Mask")
            axes[2].imshow(skel, cmap="gray");            axes[2].set_title("Skeleton")
            axes[3].imshow(cv2.cvtColor(blob_overlay, cv2.COLOR_BGR2RGB)); axes[3].set_title("Cut Lines")
            for ax in axes: ax.axis("off")
            plt.tight_layout(); plt.show()

    if show_debug:
        fig, axes = plt.subplots(1, 2, figsize=(10, 4))
        axes[0].imshow(cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB));     axes[0].set_title("Original")
        axes[1].imshow(cv2.cvtColor(final_overlay, cv2.COLOR_BGR2RGB)); axes[1].set_title("Backbone Cut Lines")
        for ax in axes: ax.axis("off")
        plt.tight_layout(); plt.show()

    return mask, final_overlay


# ── Run backbone segmentation on sampled cells ──
for i, (x, y, w, h) in enumerate(picked_rects, 1):
    crop = rotated_img[y:y+h, x:x+w]
    print(f"Cell {i} (backbone):")
    mask, overlay = segment_with_backbone(crop, show_debug=SHOW_DEBUG)